# BigQuery TPC-DS 1G — View Layer Builder

Creates a clean `mstr_view` dataset in the `mstr-tpc` BigQuery project with one view per TPC-DS table.  
The MSTR project will connect to these views instead of the raw TPC-DS tables.

In [1]:
import json
import yaml
from pprint import pprint
from google.cloud import bigquery
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from mstr_robotics.bq_connector import get_bq_client, get_bq_config

## Config

In [2]:
# All settings live in config/bq_config.yml
cfg = get_bq_config()

BQ_PROJECT   = cfg["project"]        # mstr-tpc  — where views are written
SRC_PROJECT  = cfg["src_project"]    # bigquery-public-data — where raw tables live
SRC_DATASET  = cfg["src_dataset"]    # tpc_ds_1g
VIEW_DATASET = cfg["view_dataset"]   # mstr_view
LOCATION     = cfg.get("location", "US")

print(f"view project : {BQ_PROJECT}")
print(f"src  project : {SRC_PROJECT}")
print(f"src  dataset : {SRC_DATASET}")
print(f"view dataset : {VIEW_DATASET}")
print(f"location     : {LOCATION}")

view project : mstr-tpc
src  project : bigquery-public-data
src  dataset : tpc_ds_1g
view dataset : mstr_view
location     : US


## Connect to BigQuery

In [3]:
# get_bq_client() tries:
#   1. config/bq_sa_key.json  (service account)
#   2. config/bq_user_token.json (cached browser token)
#   3. raises a clear error with setup instructions
#
# First time on a machine without a SA key → run login_browser() once instead:
#   from mstr_robotics.bq_connector import login_browser
#   bq = login_browser()   # opens browser, caches token, then get_bq_client() works forever

from mstr_robotics.bq_connector import get_bq_client, login_browser

bq = login_browser()
print(f"Client project: {bq.project}")

[bq_connector] using cached user token
[bq_connector] connected to project=mstr-tpc
Client project: mstr-tpc


## Inspect source dataset

In [4]:
# List all tables in the source public dataset
src_tables = list(bq.list_tables(f"{SRC_PROJECT}.{SRC_DATASET}"))
src_table_names = sorted([t.table_id for t in src_tables])
print(f"Found {len(src_table_names)} tables in {SRC_PROJECT}.{SRC_DATASET}:")
for t in src_table_names:
    print(f"  {t}")

Found 25 tables in bigquery-public-data.tpc_ds_1g:
  call_center
  catalog_page
  catalog_returns
  catalog_sales
  customer
  customer_address
  customer_demographics
  date_dim
  dbgen_version
  household_demographics
  income_band
  inventory
  item
  promotion
  reason
  ship_mode
  store
  store_returns
  store_sales
  time_dim
  warehouse
  web_page
  web_returns
  web_sales
  web_site


In [5]:
def fetch_schemas(bq_client, project, dataset, table_names):
    schemas = {}
    for tbl in table_names:
        ref = bq_client.get_table(f"{project}.{dataset}.{tbl}")
        schemas[tbl] = ref.schema
    return schemas

src_schemas = fetch_schemas(bq, SRC_PROJECT, SRC_DATASET, src_table_names)
print("Schemas loaded.")

for tbl, fields in list(src_schemas.items())[:3]:
    print(f"\n{tbl}: {[f.name for f in fields]}")

Schemas loaded.

call_center: ['cc_call_center_sk', 'cc_call_center_id', 'cc_rec_start_date', 'cc_rec_end_date', 'cc_closed_date_sk', 'cc_open_date_sk', 'cc_name', 'cc_class', 'cc_employees', 'cc_sq_ft', 'cc_hours', 'cc_manager', 'cc_mkt_id', 'cc_mkt_class', 'cc_mkt_desc', 'cc_market_manager', 'cc_division', 'cc_division_name', 'cc_company', 'cc_company_name', 'cc_street_number', 'cc_street_name', 'cc_street_type', 'cc_suite_number', 'cc_city', 'cc_county', 'cc_state', 'cc_zip', 'cc_country', 'cc_gmt_offset', 'cc_tax_percentage']

catalog_page: ['cp_catalog_page_sk', 'cp_catalog_page_id', 'cp_start_date_sk', 'cp_end_date_sk', 'cp_department', 'cp_catalog_number', 'cp_catalog_page_number', 'cp_description', 'cp_type']

catalog_returns: ['cr_returned_date_sk', 'cr_returned_time_sk', 'cr_item_sk', 'cr_refunded_customer_sk', 'cr_refunded_cdemo_sk', 'cr_refunded_hdemo_sk', 'cr_refunded_addr_sk', 'cr_returning_customer_sk', 'cr_returning_cdemo_sk', 'cr_returning_hdemo_sk', 'cr_returning_addr

## Create view dataset

## Load table classification (fact vs lu)

In [ ]:
with open(r"..\config\tpcds_table_classification.json", "r") as f:
    tbl_cls = json.load(f)

fact_tables = set(tbl_cls["fact_tables"].keys())
lu_tables   = set(tbl_cls["lu_tables"].keys())

print("Fact tables :", sorted(fact_tables))
print("LU tables   :", sorted(lu_tables))
print("Unclassified:", sorted(set(src_table_names) - fact_tables - lu_tables))

In [6]:
def ensure_dataset(bq_client, project, dataset_id, location):
    full_id = f"{project}.{dataset_id}"
    ds = bigquery.Dataset(full_id)
    ds.location = location
    ds = bq_client.create_dataset(ds, exists_ok=True)
    print(f"Dataset ready: {full_id}")
    return ds

ensure_dataset(bq, BQ_PROJECT, VIEW_DATASET, LOCATION)

Dataset ready: mstr-tpc.mstr_view


Dataset(DatasetReference('mstr-tpc', 'mstr_view'))

## Build view SQL helpers

In [7]:
def build_view_sql(view_project, src_project, src_dataset, view_dataset, table_name, schema_fields):
    """
    CREATE OR REPLACE VIEW in view_project.view_dataset pointing at
    src_project.src_dataset.table_name (the bigquery-public-data tables).
    """
    cols = ",\n    ".join([f"`{f.name}`" for f in schema_fields])
    sql = (
        f"CREATE OR REPLACE VIEW `{view_project}.{view_dataset}.{table_name}` AS\n"
        f"SELECT\n"
        f"    {cols}\n"
        f"FROM\n"
        f"    `{src_project}.{src_dataset}.{table_name}`"
    )
    return sql

# Preview one view
sample_tbl = src_table_names[0]
print(build_view_sql(BQ_PROJECT, SRC_PROJECT, SRC_DATASET, VIEW_DATASET,
                     sample_tbl, src_schemas[sample_tbl]))

CREATE OR REPLACE VIEW `mstr-tpc.mstr_view.call_center` AS
SELECT
    `cc_call_center_sk`,
    `cc_call_center_id`,
    `cc_rec_start_date`,
    `cc_rec_end_date`,
    `cc_closed_date_sk`,
    `cc_open_date_sk`,
    `cc_name`,
    `cc_class`,
    `cc_employees`,
    `cc_sq_ft`,
    `cc_hours`,
    `cc_manager`,
    `cc_mkt_id`,
    `cc_mkt_class`,
    `cc_mkt_desc`,
    `cc_market_manager`,
    `cc_division`,
    `cc_division_name`,
    `cc_company`,
    `cc_company_name`,
    `cc_street_number`,
    `cc_street_name`,
    `cc_street_type`,
    `cc_suite_number`,
    `cc_city`,
    `cc_county`,
    `cc_state`,
    `cc_zip`,
    `cc_country`,
    `cc_gmt_offset`,
    `cc_tax_percentage`
FROM
    `bigquery-public-data.tpc_ds_1g.call_center`


## Create all views

In [8]:
def create_views(bq_client, view_project, src_project, src_dataset, view_dataset, schemas):
    results = []
    for table_name, fields in schemas.items():
        sql = build_view_sql(view_project, src_project, src_dataset, view_dataset,
                             table_name, fields)
        try:
            bq_client.query(sql).result()
            results.append({"table": table_name, "status": "OK"})
            print(f"  ✓  {table_name}")
        except Exception as e:
            results.append({"table": table_name, "status": "ERROR", "error": str(e)})
            print(f"  ✗  {table_name}: {e}")
    return results

print(f"Creating views in `{BQ_PROJECT}.{VIEW_DATASET}` → source: `{SRC_PROJECT}.{SRC_DATASET}` ...")
view_results = create_views(bq, BQ_PROJECT, SRC_PROJECT, SRC_DATASET, VIEW_DATASET, src_schemas)

ok  = [r for r in view_results if r["status"] == "OK"]
err = [r for r in view_results if r["status"] == "ERROR"]
print(f"\nDone — {len(ok)} views created, {len(err)} errors.")
if err:
    pprint(err)

Creating views in `mstr-tpc.mstr_view` → source: `bigquery-public-data.tpc_ds_1g` ...
  ✓  call_center
  ✓  catalog_page
  ✓  catalog_returns
  ✓  catalog_sales
  ✓  customer
  ✓  customer_address
  ✓  customer_demographics
  ✓  date_dim
  ✓  dbgen_version
  ✓  household_demographics
  ✓  income_band
  ✓  inventory
  ✓  item
  ✓  promotion
  ✓  reason
  ✓  ship_mode
  ✓  store
  ✓  store_returns
  ✓  store_sales
  ✓  time_dim
  ✓  warehouse
  ✓  web_page
  ✓  web_returns
  ✓  web_sales
  ✓  web_site

Done — 25 views created, 0 errors.


## Verify views

In [9]:
# Confirm views are visible
created_views = list(bq.list_tables(f"{BQ_PROJECT}.{VIEW_DATASET}"))
print(f"{len(created_views)} views in `{VIEW_DATASET}`:")
for v in sorted(created_views, key=lambda x: x.table_id):
    print(f"  {v.table_id}  [{v.table_type}]")

25 views in `mstr_view`:
  call_center  [VIEW]
  catalog_page  [VIEW]
  catalog_returns  [VIEW]
  catalog_sales  [VIEW]
  customer  [VIEW]
  customer_address  [VIEW]
  customer_demographics  [VIEW]
  date_dim  [VIEW]
  dbgen_version  [VIEW]
  household_demographics  [VIEW]
  income_band  [VIEW]
  inventory  [VIEW]
  item  [VIEW]
  promotion  [VIEW]
  reason  [VIEW]
  ship_mode  [VIEW]
  store  [VIEW]
  store_returns  [VIEW]
  store_sales  [VIEW]
  time_dim  [VIEW]
  warehouse  [VIEW]
  web_page  [VIEW]
  web_returns  [VIEW]
  web_sales  [VIEW]
  web_site  [VIEW]


In [10]:
# Smoke-test: row count on call_center via view
smoke = bq.query(
    f"SELECT COUNT(*) AS cnt FROM `{BQ_PROJECT}.{VIEW_DATASET}.call_center`"
).result()
for row in smoke:
    print(f"call_center row count via view: {row.cnt:,}")

call_center row count via view: 6


## Export view schema catalogue (for MSTR project builder)

In [11]:
view_catalogue = {}
for table_name, fields in src_schemas.items():
    view_catalogue[table_name] = [
        {"name": f.name, "bq_type": f.field_type, "mode": f.mode}
        for f in fields
    ]

catalogue_path = r"..\config\bq_tpcds_view_catalogue.json"
with open(catalogue_path, "w", encoding="utf-8") as fp:
    json.dump(
        {"view_project": BQ_PROJECT,
         "src_project":  SRC_PROJECT,
         "src_dataset":  SRC_DATASET,
         "view_dataset": VIEW_DATASET,
         "tables": view_catalogue},
        fp, indent=2
    )

print(f"Catalogue saved → {catalogue_path}")
print(f"Tables ({len(view_catalogue)}): {list(view_catalogue.keys())}")

Catalogue saved → ..\config\bq_tpcds_view_catalogue.json
Tables (25): ['call_center', 'catalog_page', 'catalog_returns', 'catalog_sales', 'customer', 'customer_address', 'customer_demographics', 'date_dim', 'dbgen_version', 'household_demographics', 'income_band', 'inventory', 'item', 'promotion', 'reason', 'ship_mode', 'store', 'store_returns', 'store_sales', 'time_dim', 'warehouse', 'web_page', 'web_returns', 'web_sales', 'web_site']
